In [48]:
# ==========================================
# Import Libraries
# ==========================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error
)

from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

from statsmodels.tsa.statespace.sarimax import SARIMAX

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [49]:
# ==========================================
# Load Dataset
# ==========================================

df = pd.read_excel(r"C:\Users\Amey\Desktop\Amey\Python\100\Preprocessing\100_Pre_done_Combined.xlsx")

print(df.shape)

df.head()

(21349, 17)


,Material,SLoc,Quantity,Pstng Date,order,Equipment,Technician name,Year,Tavg,Tmax,Tmin,RH,Month,Season,Delta_T,Region,Location
0,100,5023,-1,2020-01-04,48550533,10930429,Anil Sharma,2020,12.94,21.27,7.26,58.52,1,Winter,14.01,North1,Haryana
1,100,5024,-1,2020-01-06,48556766,10844557,Jogendra Singh,2020,14.94,22.58,9.03,56.47,1,Winter,13.55,North1,Jaipur
2,100,5030,-1,2020-01-06,48550093,10517828,Himanshu Kushwaha,2020,13.70,21.92,8.25,55.23,1,Winter,13.67,North1,Delhi
3,100,5002,-1,2020-01-06,48550185,10519283,Prasad Chokhat,2020,22.83,30.49,16.34,68.83,1,Winter,14.15,West1,Navi Mumbai
4,100,5044,-1,2020-01-06,48554032,10836205,Joby Varghese,2020,28.71,32.68,25.08,68.80,1,Winter,7.60,South,Cochin


In [50]:
# ==========================================
# Dataset Overview
# ==========================================

print("Shape :", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isna().sum())

print("\nUnique SLocs:")
print(df['SLoc'].nunique())

df.info()

Shape : (21349, 17)

Columns:
['Material', 'SLoc', 'Quantity', 'Pstng Date', 'order', 'Equipment', 'Technician name', 'Year', 'Tavg', 'Tmax', 'Tmin', 'RH', 'Month', 'Season', 'Delta_T', 'Region', 'Location']

Missing Values:
Material           0
SLoc               0
Quantity           0
Pstng Date         0
order              0
Equipment          0
Technician name    0
Year               0
Tavg               0
Tmax               0
Tmin               0
RH                 0
Month              0
Season             0
Delta_T            0
Region             0
Location           0
dtype: int64

Unique SLocs:
76
<class 'pandas.DataFrame'>
RangeIndex: 21349 entries, 0 to 21348
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Material         21349 non-null  int64  
 1   SLoc             21349 non-null  object 
 2   Quantity         21349 non-null  int64  
 3   Pstng Date       21349 non-null  str    
 4   order   

In [51]:
# ==========================================
# Convert Date Columns
# ==========================================

df['Pstng Date'] = pd.to_datetime(df['Pstng Date'])

df['Year'] = df['Pstng Date'].dt.year
df['Month'] = df['Pstng Date'].dt.month

df.head()

,Material,SLoc,Quantity,Pstng Date,order,Equipment,Technician name,Year,Tavg,Tmax,Tmin,RH,Month,Season,Delta_T,Region,Location
0,100,5023,-1,2020-01-04,48550533,10930429,Anil Sharma,2020,12.94,21.27,7.26,58.52,1,Winter,14.01,North1,Haryana
1,100,5024,-1,2020-01-06,48556766,10844557,Jogendra Singh,2020,14.94,22.58,9.03,56.47,1,Winter,13.55,North1,Jaipur
2,100,5030,-1,2020-01-06,48550093,10517828,Himanshu Kushwaha,2020,13.70,21.92,8.25,55.23,1,Winter,13.67,North1,Delhi
3,100,5002,-1,2020-01-06,48550185,10519283,Prasad Chokhat,2020,22.83,30.49,16.34,68.83,1,Winter,14.15,West1,Navi Mumbai
4,100,5044,-1,2020-01-06,48554032,10836205,Joby Varghese,2020,28.71,32.68,25.08,68.80,1,Winter,7.60,South,Cochin


In [52]:
# ==========================================
# Monthly Consumption Aggregation
# ==========================================

df['Consumption'] = df['Quantity'].abs()

monthly_consumption = (
    df.groupby(
        [
            'SLoc',
            pd.Grouper(
                key='Pstng Date',
                freq='MS'
            )
        ]
    )['Consumption']
    .sum()
    .reset_index()
)

monthly_consumption.head()

,SLoc,Pstng Date,Consumption
0,5001,2020-01-01,19
1,5001,2020-02-01,21
2,5001,2020-03-01,15
3,5001,2020-05-01,5
4,5001,2020-06-01,23


In [53]:
# ==========================================
# Create Complete Monthly Timeline
# ==========================================

all_months = pd.date_range(
    start='2020-01-01',
    end='2026-12-01',
    freq='MS'
)

all_slocs = monthly_consumption['SLoc'].unique()

full_index = pd.MultiIndex.from_product(
    [all_slocs, all_months],
    names=['SLoc', 'Pstng Date']
)

monthly_consumption = (
    monthly_consumption
    .set_index(['SLoc', 'Pstng Date'])
    .reindex(full_index)
    .reset_index()
)

monthly_consumption['Consumption'] = (
    monthly_consumption['Consumption']
    .fillna(0)
)

monthly_consumption.head()

,SLoc,Pstng Date,Consumption
0,5001,2020-01-01,19.0
1,5001,2020-02-01,21.0
2,5001,2020-03-01,15.0
3,5001,2020-04-01,0.0
4,5001,2020-05-01,5.0


In [54]:
# ==========================================
# Aggregate Monthly Weather Data
# ==========================================

weather_monthly = (
    df.groupby(
        [
            'SLoc',
            pd.Grouper(
                key='Pstng Date',
                freq='MS'
            )
        ]
    )
    .agg(
        {
            'Tavg':'mean',
            'Tmax':'mean',
            'Tmin':'mean',
            'RH':'mean',
            'Delta_T':'mean'
        }
    )
    .reset_index()
)

weather_monthly.head()
monthly = monthly_consumption.merge(
    weather_monthly,
    on=['SLoc','Pstng Date'],
    how='left'
)

monthly.head()

,SLoc,Pstng Date,Consumption,Tavg,Tmax,Tmin,RH,Delta_T
0,5001,2020-01-01,19.0,21.661579,29.081579,15.856842,67.084211,13.224737
1,5001,2020-02-01,21.0,26.452381,35.247619,20.002381,45.686667,15.245238
2,5001,2020-03-01,15.0,25.628667,33.601333,19.414000,52.732000,14.187333
3,5001,2020-04-01,0.0,NaN,NaN,NaN,NaN,NaN
4,5001,2020-05-01,5.0,32.292000,39.894000,26.452000,57.042000,13.442000


In [55]:
# ==========================================
# Calendar Features
# ==========================================

monthly['Year'] = monthly['Pstng Date'].dt.year
monthly['Month'] = monthly['Pstng Date'].dt.month

monthly['Quarter'] = (
    monthly['Pstng Date']
    .dt
    .quarter
)

monthly['month_sin'] = np.sin(
    2*np.pi*monthly['Month']/12
)

monthly['month_cos'] = np.cos(
    2*np.pi*monthly['Month']/12
)

monthly.head()

,SLoc,Pstng Date,Consumption,Tavg,Tmax,Tmin,RH,Delta_T,Year,Month,Quarter,month_sin,month_cos
0,5001,2020-01-01,19.0,21.661579,29.081579,15.856842,67.084211,13.224737,2020,1,1,0.500000,8.660254e-01
1,5001,2020-02-01,21.0,26.452381,35.247619,20.002381,45.686667,15.245238,2020,2,1,0.866025,5.000000e-01
2,5001,2020-03-01,15.0,25.628667,33.601333,19.414000,52.732000,14.187333,2020,3,1,1.000000,6.123234e-17
3,5001,2020-04-01,0.0,NaN,NaN,NaN,NaN,NaN,2020,4,2,0.866025,-5.000000e-01
4,5001,2020-05-01,5.0,32.292000,39.894000,26.452000,57.042000,13.442000,2020,5,2,0.500000,-8.660254e-01


In [56]:
# ==========================================
# Lag Features
# ==========================================

for lag in [1,2,3,6,12]:

    monthly[f'lag_{lag}'] = (
        monthly
        .groupby('SLoc')['Consumption']
        .shift(lag)
    )

monthly.head()

,SLoc,Pstng Date,Consumption,Tavg,Tmax,Tmin,RH,Delta_T,Year,Month,Quarter,month_sin,month_cos,lag_1,lag_2,lag_3,lag_6,lag_12
0,5001,2020-01-01,19.0,21.661579,29.081579,15.856842,67.084211,13.224737,2020,1,1,0.500000,8.660254e-01,NaN,NaN,NaN,NaN,NaN
1,5001,2020-02-01,21.0,26.452381,35.247619,20.002381,45.686667,15.245238,2020,2,1,0.866025,5.000000e-01,19.0,NaN,NaN,NaN,NaN
2,5001,2020-03-01,15.0,25.628667,33.601333,19.414000,52.732000,14.187333,2020,3,1,1.000000,6.123234e-17,21.0,19.0,NaN,NaN,NaN
3,5001,2020-04-01,0.0,NaN,NaN,NaN,NaN,NaN,2020,4,2,0.866025,-5.000000e-01,15.0,21.0,19.0,NaN,NaN
4,5001,2020-05-01,5.0,32.292000,39.894000,26.452000,57.042000,13.442000,2020,5,2,0.500000,-8.660254e-01,0.0,15.0,21.0,NaN,NaN


In [57]:
# ==========================================
# Rolling Features
# ==========================================

monthly['roll_mean_3'] = (
    monthly.groupby('SLoc')['Consumption']
    .transform(
        lambda x:
        x.shift(1)
         .rolling(3)
         .mean()
    )
)

monthly['roll_mean_6'] = (
    monthly.groupby('SLoc')['Consumption']
    .transform(
        lambda x:
        x.shift(1)
         .rolling(6)
         .mean()
    )
)

monthly['roll_mean_12'] = (
    monthly.groupby('SLoc')['Consumption']
    .transform(
        lambda x:
        x.shift(1)
         .rolling(12)
         .mean()
    )
)

monthly['roll_std_3'] = (
    monthly.groupby('SLoc')['Consumption']
    .transform(
        lambda x:
        x.shift(1)
         .rolling(3)
         .std()
    )
)

monthly['roll_std_6'] = (
    monthly.groupby('SLoc')['Consumption']
    .transform(
        lambda x:
        x.shift(1)
         .rolling(6)
         .std()
    )
)

monthly['roll_std_12'] = (
    monthly.groupby('SLoc')['Consumption']
    .transform(
        lambda x:
        x.shift(1)
         .rolling(12)
         .std()
    )
)

monthly['roll_max_3'] = (
    monthly.groupby('SLoc')['Consumption']
    .transform(
        lambda x:
        x.shift(1)
         .rolling(3)
         .max()
    )
)

monthly['roll_min_3'] = (
    monthly.groupby('SLoc')['Consumption']
    .transform(
        lambda x:
        x.shift(1)
         .rolling(3)
         .min()
    )
)

monthly.head()

,SLoc,Pstng Date,Consumption,Tavg,Tmax,Tmin,RH,Delta_T,Year,Month,Quarter,month_sin,month_cos,lag_1,lag_2,lag_3,lag_6,lag_12,roll_mean_3,roll_mean_6,roll_mean_12,roll_std_3,roll_std_6,roll_std_12,roll_max_3,roll_min_3
0,5001,2020-01-01,19.0,21.661579,29.081579,15.856842,67.084211,13.224737,2020,1,1,0.500000,8.660254e-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5001,2020-02-01,21.0,26.452381,35.247619,20.002381,45.686667,15.245238,2020,2,1,0.866025,5.000000e-01,19.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5001,2020-03-01,15.0,25.628667,33.601333,19.414000,52.732000,14.187333,2020,3,1,1.000000,6.123234e-17,21.0,19.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5001,2020-04-01,0.0,NaN,NaN,NaN,NaN,NaN,2020,4,2,0.866025,-5.000000e-01,15.0,21.0,19.0,NaN,NaN,18.333333,NaN,NaN,3.055050,NaN,NaN,21.0,15.0
4,5001,2020-05-01,5.0,32.292000,39.894000,26.452000,57.042000,13.442000,2020,5,2,0.500000,-8.660254e-01,0.0,15.0,21.0,NaN,NaN,12.000000,NaN,NaN,10.816654,NaN,NaN,21.0,0.0


In [58]:
# ==========================================
# Remove NaNs from Lag/Rolling Features
# ==========================================

monthly = monthly.dropna().reset_index(drop=True)

print(monthly.shape)

monthly.head()

(1424, 26)


,SLoc,Pstng Date,Consumption,Tavg,Tmax,Tmin,RH,Delta_T,Year,Month,Quarter,month_sin,month_cos,lag_1,lag_2,lag_3,lag_6,lag_12,roll_mean_3,roll_mean_6,roll_mean_12,roll_std_3,roll_std_6,roll_std_12,roll_max_3,roll_min_3
0,5001,2021-01-01,44.0,23.798077,31.343462,18.510000,61.378077,12.833462,2021,1,1,0.500000,8.660254e-01,15.0,16.0,17.0,14.0,19.0,16.000000,16.333333,15.083333,1.000000,2.503331,6.625822,17.0,15.0
1,5001,2021-02-01,17.0,25.028824,34.398824,18.077059,45.559412,16.321765,2021,2,1,0.866025,5.000000e-01,44.0,15.0,16.0,15.0,21.0,25.000000,21.333333,17.166667,16.462078,11.325487,10.667140,44.0,15.0
2,5001,2021-03-01,29.0,29.422500,38.661071,22.167143,39.315000,16.493929,2021,3,1,1.000000,6.123234e-17,17.0,44.0,15.0,21.0,15.0,25.333333,21.666667,16.833333,16.196707,11.129540,10.598742,44.0,15.0
3,5001,2021-04-01,12.0,30.856667,38.848333,24.702500,53.057500,14.145833,2021,4,2,0.866025,-5.000000e-01,29.0,17.0,44.0,17.0,0.0,30.000000,23.000000,18.000000,13.527749,11.506520,11.135529,44.0,17.0
4,5001,2021-05-01,11.0,30.438182,35.420909,26.668182,67.158182,8.752727,2021,5,2,0.500000,-8.660254e-01,12.0,29.0,17.0,16.0,5.0,19.333333,22.166667,19.000000,8.736895,12.188793,9.835002,29.0,12.0


In [59]:
# ==========================================
# Feature Selection
# ==========================================

FEATURES = [

    'Year',

    'month_sin',
    'month_cos',
    'Quarter',

    'lag_1',
    'lag_2',
    'lag_3',
    'lag_6',
    'lag_12',

    'roll_mean_3',
    'roll_mean_6',
    'roll_mean_12',

    'roll_std_3',
    'roll_std_6',
    'roll_std_12',

    'roll_max_3',
    'roll_min_3',

    'Tavg',
    'Tmax',
    'Tmin',
    'RH',
    'Delta_T'
]

TARGET = 'Consumption'

In [60]:
# ==========================================
# Train Test Split
# ==========================================

train = monthly[
    monthly['Pstng Date'] < '2025-01-01'
].copy()

test = monthly[
    (monthly['Pstng Date'] >= '2025-01-01') &
    (monthly['Pstng Date'] <= '2026-04-30')
].copy()

print("Train Shape :", train.shape)
print("Test Shape :", test.shape)

Train Shape : (1065, 26)
Test Shape : (359, 26)


In [61]:
# ==========================================
# Create X and y
# ==========================================

X_train = train[FEATURES]
y_train = train[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

In [62]:
# ==========================================
# Evaluation Metrics
# ==========================================

def wmape(y_true, y_pred):

    return (
        np.sum(np.abs(y_true - y_pred))
        /
        np.sum(np.abs(y_true))
    ) * 100


def evaluate_model(name, y_true, y_pred):

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    mape = mean_absolute_percentage_error(
        y_true,
        y_pred
    ) * 100

    wmape_score = wmape(
        y_true,
        y_pred
    )

    return pd.DataFrame({

        'Model':[name],
        'MAE':[mae],
        'RMSE':[rmse],
        'MAPE':[mape],
        'WMAPE':[wmape_score]

    })

In [63]:
# ==========================================
# XGBoost
# ==========================================

xgb_model = XGBRegressor(

    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,

    random_state=42
)

xgb_model.fit(
    X_train,
    y_train
)

test['XGB_Pred'] = xgb_model.predict(
    X_test
)

xgb_metrics = evaluate_model(
    'XGBoost',
    y_test,
    test['XGB_Pred']
)

xgb_metrics

,Model,MAE,RMSE,MAPE,WMAPE
0,XGBoost,5.75891,12.912838,75.544568,29.061693


In [64]:
# ==========================================
# CatBoost
# ==========================================

cat_model = CatBoostRegressor(

    iterations=500,
    learning_rate=0.05,
    depth=6,

    verbose=0,

    random_state=42
)

cat_model.fit(
    X_train,
    y_train
)

test['CAT_Pred'] = cat_model.predict(
    X_test
)

cat_metrics = evaluate_model(
    'CatBoost',
    y_test,
    test['CAT_Pred']
)

cat_metrics

,Model,MAE,RMSE,MAPE,WMAPE
0,CatBoost,5.396943,12.773978,73.721563,27.235067


In [65]:
# ==========================================
# LightGBM
# ==========================================

lgbm_model = LGBMRegressor(

    n_estimators=500,
    learning_rate=0.05,

    random_state=42
)

lgbm_model.fit(
    X_train,
    y_train
)

test['LGBM_Pred'] = lgbm_model.predict(
    X_test
)

lgbm_metrics = evaluate_model(
    'LightGBM',
    y_test,
    test['LGBM_Pred']
)

lgbm_metrics

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000205 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2900
[LightGBM] [Info] Number of data points in the train set: 1065, number of used features: 22
[LightGBM] [Info] Start training from score 12.548357


,Model,MAE,RMSE,MAPE,WMAPE
0,LightGBM,6.379013,15.05563,77.859577,32.190972


In [66]:
# ==========================================
# Random Forest
# ==========================================

rf_model = RandomForestRegressor(

    n_estimators=500,

    random_state=42,

    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

test['RF_Pred'] = rf_model.predict(
    X_test
)

rf_metrics = evaluate_model(
    'RandomForest',
    y_test,
    test['RF_Pred']
)

rf_metrics

,Model,MAE,RMSE,MAPE,WMAPE
0,RandomForest,5.583538,13.203972,84.13338,28.176694


In [67]:
# ==========================================
# Compare Models
# ==========================================

model_metrics = pd.concat([

    xgb_metrics,
    cat_metrics,
    lgbm_metrics,
    rf_metrics

])

model_metrics.sort_values(
    'WMAPE'
)

,Model,MAE,RMSE,MAPE,WMAPE
0,CatBoost,5.396943,12.773978,73.721563,27.235067
0,RandomForest,5.583538,13.203972,84.133380,28.176694
0,XGBoost,5.758910,12.912838,75.544568,29.061693
0,LightGBM,6.379013,15.055630,77.859577,32.190972
